In [19]:
from pathlib import Path
import os
import json
from pypdf import PdfReader
import pandas as pd
import sqlite3

In [20]:
home_path = Path("/home/daniel/code/exercises_classifier/")

In [21]:
with open(home_path / "topics.json", mode="r", encoding="utf-8") as file:
    topics_dict = json.load(file)

In [22]:
def extract_info_from_pdf(pdf_file):
    reader = PdfReader(pdf_file)
    text = reader.pages[0].extract_text()
    text_lines = [line.strip() for line in text.split("\n") if line.strip()][3:]
    
    return text_lines

In [30]:
def parse_lines(text_lines):
    
    year = int(text_lines[0])
    subject = text_lines[1].lower()
    topic = text_lines[2].split(":")[1].strip().lower()
    topic = topics_dict[topic]

    exercises = []
    for line in text_lines[3:]:
        line = line[2:]  # remove "• "
        components = line.split(', ')
        exam = components[0]
        exercise = ', '.join(components[1:])
        exercises.append(f"{subject}|{topic}|{year}|{exam}|{exercise}")

    return exercises 

In [25]:
os.chdir("./pdf/química")
os.getcwd()

'/home/daniel/code/exercises_classifier/pdf/química'

In [26]:
Path.cwd()

PosixPath('/home/daniel/code/exercises_classifier/pdf/química')

In [27]:
pdf_files_path = home_path / "pdf/química"
folders = [folder for folder in pdf_files_path.iterdir() if folder.name != ".DS_Store"]
folders

[PosixPath('/home/daniel/code/exercises_classifier/pdf/química/ácido base'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/redox'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/equilibrio químico'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/solubilidad'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/reactividad orgánica'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/formulación'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/cantidad química'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/átomo'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/termoquímica'),
 PosixPath('/home/daniel/code/exercises_classifier/pdf/química/enlace químico')]

In [36]:
with open(home_path / "classifications.csv", mode="w", encoding="utf-8") as f:
    headers =  "asignatura|tema|año|convocatoria|ejercicio|tipo_ejercicio"
    f.write(headers + "\n")
    for folder in folders:
        for file in folder.iterdir():
            if file.suffix == ".pdf":
                try:
                    lines = extract_info_from_pdf(file)
                    exercises = parse_lines(lines)
                    for e in exercises:
                        f.write(e + "\n")
                except Exception as e:
                    print(f"Error: {e}. Error parsing file: {file}")
                    continue

In [37]:
os.chdir(home_path)

In [38]:
df = pd.read_csv("classifications.csv", sep="|")

In [39]:
df

,asignatura,tema,año,convocatoria,ejercicio,tipo_ejercicio
0,química,ácido base,2012,Junio,"Ejercicio 6, Opción A",NaN
1,química,ácido base,2012,Junio,"Ejercicio 4, Opción B",NaN
2,química,ácido base,2012,Reserva 1,"Ejercicio 4, Opción A",NaN
3,química,ácido base,2012,Reserva 2,"Ejercicio 5, Opción A",NaN
4,química,ácido base,2012,Reserva 2,"Ejercicio 4, Opción B",NaN
...,...,...,...,...,...,...
1808,química,enlace químico,2002,Reserva 1,"Ejercicio 3, Opción B",NaN
1809,química,enlace químico,2002,Reserva 2,"Ejercicio 3, Opción A",NaN
1810,química,enlace químico,2002,Reserva 3,"Ejercicio 2, Opción B",NaN
1811,química,enlace químico,2002,Reserva 4,"Ejercicio 3, Opción B",NaN


In [41]:
conn = sqlite3.connect(home_path / "database.db")

In [42]:
conn.execute("""
            CREATE TABLE IF NOT EXISTS ejercicios (
                id           INTEGER PRIMARY KEY AUTOINCREMENT,
                asignatura   TEXT    NOT NULL,
                tema         TEXT    NOT NULL,
                año          INTEGER NOT NULL,
                convocatoria TEXT    NOT NULL,
                ejercicio    TEXT    NOT NULL,
                tipo_ejercicio TEXT  NOT NULL
            )
        """)
conn.commit()

In [43]:
df.to_sql("ejercicios", conn, if_exists="replace", index = False)

1813

In [44]:
conn = sqlite3.connect("database.db")

# get the dataframe from all available data in db
df = pd.read_sql("select * from ejercicios ;", conn)

In [45]:
df.head()

,asignatura,tema,año,convocatoria,ejercicio,tipo_ejercicio
0,química,ácido base,2012,Junio,"Ejercicio 6, Opción A",None
1,química,ácido base,2012,Junio,"Ejercicio 4, Opción B",None
2,química,ácido base,2012,Reserva 1,"Ejercicio 4, Opción A",None
3,química,ácido base,2012,Reserva 2,"Ejercicio 5, Opción A",None
4,química,ácido base,2012,Reserva 2,"Ejercicio 4, Opción B",None


In [1]:
selected_subject = "química"
selected_topic = "ácido base"
selected_year = 2025

In [25]:
df.loc[df.asignatura == selected_subject, "tema"].unique()

<ArrowStringArray>
[          'ácido base',                'redox',   'equilibrio químico',
          'solubilidad', 'reactividad orgánica',          'formulación',
     'cantidad química',                'átomo',         'termoquímica',
       'enlace químico']
Length: 10, dtype: str

In [15]:
mask = (df.asignatura == selected_subject) & \
    (df.tema == selected_topic) & \
    (df.año == selected_year)

exercises_df = df[mask]

In [17]:
df

,asignatura,tema,año,convocatoria,ejercicio,tipo_ejercicio
0,química,ácido base,2012,Septiembre,Ejercicio 5 Opción B,None
1,química,ácido base,2011,Septiembre,Ejercicio 4 Opción B,None
2,química,ácido base,2009,Septiembre,Ejercicio 4 Opción B,None
3,química,ácido base,2014,Septiembre,Ejercicio 5 Opción B,None
4,química,ácido base,2024,Julio,Ejercicio C3,None
...,...,...,...,...,...,...
228,química,enlace químico,2014,Septiembre,Ejercicio 2 Opción B,None
229,química,enlace químico,2012,Septiembre,Ejercicio 2 Opción B,None
230,química,enlace químico,2019,Septiembre,Ejercicio 2 Opción B,None
231,química,enlace químico,2022,Julio,Ejercicio B3,None


In [11]:
exercises_df.apply(lambda row : row.loc["convocatoria"], axis=1)

9    Reserva 2
dtype: str

In [14]:
df.loc[df.asignatura == selected_subject, "tema"].unique()

<ArrowStringArray>
[          'ácido base',                'redox',   'equilibrio químico',
          'solubilidad', 'reactividad orgánica',          'formulación',
     'cantidad química',                'átomo',         'termoquímica',
       'enlace químico']
Length: 10, dtype: str